In [1]:
# Создать папку eprime вручную
# Закинуть в папку pX_verbs.xlsx и pX_protocol.docx
# Поменять номер в названии файла pX_verbs.xlsx в зависимости от номера участника
# Поменять index на Index в coordinates_MNI.xlsx

import re
import os
import pandas as pd
import json
import os
from copy import copy

import openpyxl
import random
import tqdm
import shutil
import docx
from docx import Document

# Координаты XML

In [2]:
def make_targets(df, folder):
    with open(os.path.join(folder, 'res', 'new_targets.xml'), 'w') as file:
        file.write('<?xml version="1.0" encoding="UTF-8"?>\n\
<EntryTargetList coordinateSpace="RAS">\n\
    <!--All positions are saved in the coordinate system of the corresponding medical data.\n\
NIfTI image data are recommended using RAS system (x-axis increases from the left hand side to the right hand side of the patient,\n\
y-axis increases from the posterior side to the anterior side of the patient and z-axis increases from the feet toward the head of the patient).-->\n')
        for idx, row in df.iterrows():
            new_line = f'<Entry alwaysVisible="true" index="{str(row["Index"])}" selected="false" skip="false">\n\
            <Marker color="#00ff00" description="" drawAura="false"\n\
            radius="4.0" set="true">\n\
            <ColVec3D data0=""\n\
            data1="" data2=""/>\n\
            </Marker>\n\
            </Entry> \n\
            <Target alwaysVisible="true" index="{str(row["Index"])}" selected="false" skip="false">\n\
            <Marker description="{row["Region"]}" drawAura="true"\n\
            radius="4.0" set="true">\n\
            <ColVec3D data0="{str(round(row["x"], 3)).replace(",", ".")}" \
            data1="{str(round(row["y"], 3)).replace(",", ".")}" \
            data2="{str(round(row["z"], 3)).replace(",", ".")}"/>\n\
            </Marker>\n\
            </Target>\n\
            <Rotation angle="{str(row["angle"])}" index="{str(row["Index"])}" used="true">\n\
            <RotationReference>\n\
            <ColVec3D data0="0.0" data1="0.0" data2="-1.0"/>\n\
            </RotationReference>\n\
            </Rotation>'
            file.write(new_line+ '\n')
        file.write('</EntryTargetList>')


# Стимулы

### Рандомизация и подготовка протокола

In [3]:
def make_session(p_ID, PARTICIPANT_FOLDER, regions, max_trials=60):
    reserve_size = max_trials - 46
    all_protocol = []
    
    participant_info_lines = [f'Participant ID: {p_ID}']
    stimuli_df = pd.read_excel(os.path.join(PARTICIPANT_FOLDER, f'p{p_ID}_verbs.xlsx'))

    with pd.ExcelWriter(os.path.join(PARTICIPANT_FOLDER, 'res',  f'p{p_ID}_protocol.xlsx')) as writer:    
        for session_num in range(1,3):
            verbs_df = copy(regions).reset_index(drop=True).sort_values('Index')
            for i in range(max_trials - 46):
                new_index = max(verbs_df.index) + 1
                verbs_df.loc[new_index, 'Region'] = 'reserve'
                
            random_state_stim = p_ID + 100 * session_num
            random_state_stim_reserve = random_state_stim+200
            stimuli_item = stimuli_df.sample(frac=1, random_state=random_state_stim).reset_index(drop=True)
            stimuli_item_reserve = stimuli_df.sample(frac=1, random_state=random_state_stim_reserve).reset_index(drop=True)

            while (stimuli_item.shape[0] + stimuli_item_reserve.shape[0]) < max_trials * 4:
                random_state_stim_reserve += 1
                stimuli_item_reserve_additional = stimuli_df.sample(
                    frac=1, random_state=random_state_stim_reserve).reset_index(drop=True)
                stimuli_item_reserve = pd.concat([stimuli_item_reserve, stimuli_item_reserve_additional]).reset_index(drop=True)
            
            for idx in range(4):
                item_train = stimuli_item.iloc[idx*46:(idx+1)*46]
                reserve_size = max_trials - item_train.shape[0]                    
                item_train_reserve = stimuli_item_reserve.iloc[0:reserve_size]
                stimuli_item_reserve = stimuli_item_reserve[reserve_size:]
                item_train = pd.concat([item_train, item_train_reserve], ignore_index=True).reset_index()[:max_trials]
                item_list = item_train['Целевая номинация (Target name Russian)'].tolist() + [None] * (
                    max_trials - item_train.shape[0])
                verbs_df[f'Предъявление {idx + 1}'] = item_list
                verbs_df[f'Ответ {idx + 1}'] = None
                
                item_eprime = ['Weight	Nested	Procedure	Stimulus']
                for _, row in item_train.iterrows():
                    line = f'1		TrialObjects	{row["filename"]}'
                    item_eprime.append(line)
                    
                with open(os.path.join(PARTICIPANT_FOLDER, 'eprime',  f'verbs_{int(p_ID)}_{session_num}_{idx+1}.txt'), 'w') as f:
                    f.write('\n'.join(item_eprime))
                stim_df = item_train[:max_trials][['Целевая номинация (Target name Russian)', 'filename']]
                stim_df = stim_df.rename(columns = {'filename': 'Picture_filename',
                                                    'Целевая номинация (Target name Russian)': 'Stimulus'})
                item_protocol_cur = pd.concat([verbs_df[['Index', 'Region', '№']][:max_trials], stim_df], axis=1)
                item_protocol_cur['task'] = 'ActionNaming'
                item_protocol_cur['Session'] = session_num
                item_protocol_cur['round'] = idx + 1
                all_protocol.append(item_protocol_cur)
                
            verbs_df.to_excel(writer, sheet_name = f'verbs_{session_num}', index=False)

    all_protocol = pd.concat(all_protocol)
    all_protocol['pID'] = p_ID
    for column in ['Error', 'Pain', 'Response', 'Response_transcription', 
                   'Audio_filename', 'RT_start', 'RT_end', 'Comment']:
        all_protocol[column] = None
    all_protocol.to_excel(os.path.join(PARTICIPANT_FOLDER, 'res',  f'p{p_ID}_protocol_long.xlsx'), index=False)
    return all_protocol

# Participant preparation

In [4]:
def fill_word_protocol(p_ID, participant_folder):
    """
    Fill the Word protocol document with data from the Excel protocol
    """
    try:
        from docx import Document
    except ImportError:
        print("python-docx not available. Installing...")
        import sys
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "python-docx"])
        from docx import Document
    
    # Paths to files
    excel_path = os.path.join(participant_folder, 'res', f'p{p_ID}_protocol.xlsx')
    word_path = os.path.join(participant_folder, 'res', f'p{p_ID}_Протокол.docx')
    
    # Check if files exist
    if not os.path.exists(excel_path):
        print(f"Excel file not found: {excel_path}")
        return
    if not os.path.exists(word_path):
        print(f"Word file not found: {word_path}")
        return
    
    # Read Excel data from verbs_1 sheet
    try:
        excel_data = pd.read_excel(excel_path, sheet_name='verbs_1')
    except Exception as e:
        print(f"Error reading Excel file: {e}")
        return
    
    # Filter out reserve rows and create mapping by Region
    region_data = {}
    for _, row in excel_data.iterrows():
        region = row['Region']
        if region != 'reserve' and pd.notna(region):
            region_data[region] = {
                'presentation_1': row['Предъявление 1'] if pd.notna(row['Предъявление 1']) else '',
                'presentation_2': row['Предъявление 2'] if pd.notna(row['Предъявление 2']) else '',
                'presentation_3': row['Предъявление 3'] if pd.notna(row['Предъявление 3']) else '',
                'presentation_4': row['Предъявление 4'] if pd.notna(row['Предъявление 4']) else ''
            }
    
    # Load and fill Word document
    try:
        doc = Document(word_path)
        table = doc.tables[0]  # First table in document
        
        # Skip the first two rows (header and formatting rows)
        # Start from row 2 (third row, zero-indexed)
        for row_idx, row in enumerate(table.rows[2:], 2):  # Start from row 2
            if row_idx >= len(table.rows):
                break
                
            # Get region label from second cell (index 1)
            region_label = row.cells[1].text.strip()
            
            # Clean up the region label - remove any ** markers
            region_label = region_label.replace('**', '').strip()
            
            if region_label in region_data:
                data = region_data[region_label]
                
                # Fill the stimulus cells (columns 3, 5, 7, 9 - zero-indexed)
                # These are the "Проба" cells for each presentation
                row.cells[3].text = str(data['presentation_1'])
                row.cells[5].text = str(data['presentation_2'])
                row.cells[7].text = str(data['presentation_3'])
                row.cells[9].text = str(data['presentation_4'])
            else:
                print(f"Warning: Region '{region_label}' not found in Excel data")
        
        # Save the filled document
        doc.save(word_path)
        print(f"✓ Word protocol filled for participant {p_ID}")
        
    except Exception as e:
        print(f"Error filling Word document: {e}")
        import traceback
        traceback.print_exc()

In [5]:
# Modified participant preparation section
for p_ID, FIO in [(18, 'Kolesnikova')]:
    print(p_ID, FIO)
    PARTICIPANT_FOLDER = f'p{p_ID}_{FIO}'
    
    # Create all necessary directories
    os.makedirs(os.path.join(PARTICIPANT_FOLDER, 'res'), exist_ok=True)
    os.makedirs(os.path.join(PARTICIPANT_FOLDER, 'coords'), exist_ok=True)
    os.makedirs(os.path.join(PARTICIPANT_FOLDER, 'eprime'), exist_ok=True)
    
    markers_file = os.path.join(PARTICIPANT_FOLDER, 'coords', 'coordinates_MNI.xlsx')
    
    # Read markers file and handle column names
    markers = pd.read_excel(os.path.join(PARTICIPANT_FOLDER, 'coords', 'coordinates_MNI.xlsx'))
    
    # Check and fix column names
    if 'index' in markers.columns and 'Index' not in markers.columns:
        markers.rename(columns={'index': 'Index'}, inplace=True)
    
    # Select required columns
    markers = markers[['Index', 'Region', '№']].sort_values('Index')
    regions = copy(markers)
    regions['Index'] += 1
    p_protocol = make_session(p_ID, PARTICIPANT_FOLDER, regions)
    
    # Read native coordinates and handle column names
    df = pd.read_excel(os.path.join(PARTICIPANT_FOLDER, 'coords', 'coords_native.xlsx'))
    
    # Check and fix column names in df if needed
    if 'index' in df.columns and 'Index' not in df.columns:
        df.rename(columns={'index': 'Index'}, inplace=True)
    
    # Now merge the dataframes
    df = pd.merge(df, markers, on='Index').sort_values('Index')
    make_targets(df, PARTICIPANT_FOLDER)

    shutil.copy('pX_protocol.docx', os.path.join(PARTICIPANT_FOLDER, 'res', f'p{p_ID}_Протокол.docx'))
    
    # Fill the Word protocol after copying
    fill_word_protocol(p_ID, PARTICIPANT_FOLDER)

18 Kolesnikova
✓ Word protocol filled for participant 18


# All participants

In [6]:
DATA_PATH = './'

for p_ID in tqdm.tqdm(range(3)):
    PARTICIPANT_FOLDER = f'data_participants/p{p_ID}_FIO'
    if os.path.exists(PARTICIPANT_FOLDER):
        shutil.rmtree(PARTICIPANT_FOLDER, ignore_errors=True)
    os.mkdir(PARTICIPANT_FOLDER)
    os.mkdir(os.path.join(PARTICIPANT_FOLDER, 'res'))
    os.mkdir(os.path.join(PARTICIPANT_FOLDER, 'coords'))
    os.mkdir(os.path.join(PARTICIPANT_FOLDER, 'eprime'))
    shutil.copy(os.path.join(DATA_PATH, f'pX_verbs.xlsx'),
                    os.path.join(PARTICIPANT_FOLDER, f'p{p_ID}_verbs.xlsx'))
    for file in ['coordinates_MNI.xlsx', 'get_orig_coord5.m', 'rotateBrain.m', 'transfer.m']:
        shutil.copy(os.path.join(DATA_PATH, file),
                    os.path.join(PARTICIPANT_FOLDER, 'coords', file))
    shutil.copy(os.path.join(DATA_PATH, 'pX_protocol.docx'),
               os.path.join(PARTICIPANT_FOLDER, 'res', f'p{p_ID}_Протокол.docx'))




100%|██████████| 3/3 [00:00<00:00, 87.05it/s]
